# 05. 시연: 통화 하나 넣고 증상 예측 (GPU 권장, CPU도 가능)

04에서 저장한 **정리 모델**로 통화 하나의 증상을 예측한다.
- 방법 1: Val에서 번호로 고르기 (정답 라벨과 비교)
- 방법 2: 원본 JSON 파일 하나 올리기 (`utterances` 목록이 있는 파일)

| 단계 | 하는 일 | 쓰는 함수 |
|---|---|---|
| 0 | 준비, 모델 불러오기 | `infer.load_model` |
| 1 | Val에서 골라 예측 | `infer.predict_dialogue` |
| 2 | JSON 파일 올려 예측 | `infer.predict_dialogue` |

## 0. 준비

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import json
import pandas as pd
from src import config
from src.data import load_labels, symptom_matrix
from src.text import show_dialogue
from src.infer import load_model, predict_dialogue

model, tok, meta = load_model(f"{config.MODEL_DIR}/정리모델.pt")
_, va_df = load_labels()
print("증상:", meta["labels"])

## 1. Val에서 골라 예측

In [ ]:
IDX = 0      # 바꿔 가며 보기 (0 ~ Val 대화 수-1)
row = va_df.iloc[IDX]
print(row["file_name"])
print(show_dialogue(row["utterances"])[:2000])
res = predict_dialogue(row["utterances"], model, tok, meta)
y = dict(zip(meta["labels"], symptom_matrix([row["symptom"]], meta["labels"])[0]))
res["정답 라벨"] = res["증상"].map(y).astype(int)
display(res)

## 2. JSON 파일 올려 예측

In [ ]:
from google.colab import files
up = files.upload()
for name, raw in up.items():
    d = json.loads(raw.decode("utf-8"))
    utts = d.get("utterances") or d.get("data", {}).get("utterances")
    print(name); print(show_dialogue(utts)[:2000])
    display(predict_dialogue(utts, model, tok, meta))